# Geographic Analysis — Freedom in the World

**Notebook 08 of 08**

### Purpose

The final notebook puts the findings on a map. The overall score, its change over time, an indicator, a continent zoom and the categorical status are all visualised as interactive choropleth maps — the geographic summary of the whole pipeline.

### Scope

The maps use the dataset's own `REF_AREA` codes with Plotly's ISO-3 matching. The mapping is verified explicitly in section 1: all 197 economies render, including the two non-standard codes (XKX Kosovo, TWN Taiwan). Economies that could not be matched would be dropped silently, so the check matters — per the spec, misleading maps are to be avoided.

## Data and Inputs

| Item | Location |
|---|---|
| Long analytical dataset | `data/processed/freedom_in_world_long.csv` |

**Question this notebook answers:** *Where in the world are the patterns found in notebooks 04–07?*

**Method:** six maps, all from the reusable `create_choropleth()` helper in `src/visualizations.py`:
1. Global overall score, 2026.
2. Global overall score, 2013 (same colour scale, for direct comparison).
3. Change in overall score, 2013→2026.
4. Rule of law — the weakest category — 2026, as % of scale.
5. Africa zoom, 2026.
6. Status classification (Free / Partly Free / Not Free), 2026.

The interactive year/indicator selector promised by the spec belongs to the Streamlit dashboard (Map Explorer); the notebook fixes key year/indicator combinations and compares them.

## Setup: imports and the project root

Same bootstrap. New import: `create_choropleth()` from `src/visualizations.py`.

In [1]:
import sys
from pathlib import Path

current = Path.cwd()
while not (current / 'data' / 'raw' / 'FH_FIW_WIDEF.csv').exists():
    current = current.parent
    if current == current.parent:
        raise RuntimeError('Could not find the project root.')

if str(current) not in sys.path:
    sys.path.insert(0, str(current))

import pandas as pd

from src.data_loader import load_processed_data
from src.visualizations import create_choropleth
from src.regions import EAST_AFRICA_5

print('Imports ready.')

Imports ready.


### Interpretation

Setup ran cleanly. All maps use the processed long dataset and the shared choropleth helper.

## 1. Prepare the map data and verify the matching

**Question:** can every economy be plotted reliably?

**Method:** build the 2026 overall-score frame (Economy + REF_AREA code + Score) and count how many codes actually render on an ISO-3 choropleth.

In [2]:
import plotly.express as px

long = load_processed_data()
total = long[long['INDICATOR'] == 'FH_FIW_TOTAL'].copy()
total['Score'] = pd.to_numeric(total['Score'], errors='coerce')

d2026 = total[total['Year'] == 2026].dropna(subset=['Score'])
probe = px.choropleth(d2026, locations='REF_AREA', color='Score', locationmode='ISO-3', hover_name='Economy')
rendered = set(probe.data[0].locations)

print('Economies with a 2026 score:', len(d2026))
print('Economies rendered on the map:', len(rendered))
print('Unmatched codes:', sorted(set(d2026['REF_AREA']) - rendered))

Economies with a 2026 score: 196
Economies rendered on the map: 196
Unmatched codes: []


### Interpretation

**All 196 economies with a 2026 score render** — nothing is silently dropped. The two non-standard Data360 codes, XKX (Kosovo) and TWN (Taiwan, China), are included in Plotly's ISO-3 set, so the map is complete and trustworthy.

## 2. Map 1 — global overall score, 2026

**Question:** what does the world look like today?

**Method:** choropleth of the overall score (0–100), colour scale fixed to 0–100 so later maps are comparable.

In [3]:
fig = create_choropleth(
    d2026,
    location_col='REF_AREA',
    color_col='Score',
    title='Overall freedom score by economy, 2026',
    colorbar_label='Overall score (0-100)',
    range_color=(0, 100),
)
fig.show()

### Interpretation

The map makes the two-tier world from notebook 06 visible at a glance: a **bright band across Europe, North America and Oceania**, and a **dark belt across Africa and Asia** (with bright islands for Japan, Korea, Taiwan and Israel). The extremes are Norway and Sweden (99–100) at the top and South Sudan (0), Sudan (1) and Turkmenistan (1) at the bottom.

## 3. Map 2 — global overall score, 2013

**Question:** where has the world shifted since 2013?

**Method:** the same map for 2013 with the **identical colour scale**, so the two maps are directly comparable.

In [4]:
d2013 = total[total['Year'] == 2013].dropna(subset=['Score'])
fig = create_choropleth(
    d2013,
    location_col='REF_AREA',
    color_col='Score',
    title='Overall freedom score by economy, 2013',
    colorbar_label='Overall score (0-100)',
    range_color=(0, 100),
)
fig.show()

### Interpretation

Side by side with the 2026 map, the overall geography is recognisably the same — the two-tier structure is not new — but the 2013 map is visibly **lighter in Africa and the Americas**. Economies like Tanzania, Nicaragua and El Salvador, bright in 2013, are dark by 2026. The next map measures exactly this shift.

## 4. Map 3 — change 2013→2026

**Question:** where did freedom rise, and where did it fall?

**Method:** choropleth of each economy's change (2026 − 2013), with a diverging scale centred on zero (−40 to +40): red = decline, blue = improvement.

In [5]:
change = d2013[['REF_AREA', 'Economy', 'Score']].merge(
    d2026[['REF_AREA', 'Score']], on='REF_AREA', suffixes=('_2013', '_2026')
)
change['change'] = (change['Score_2026'] - change['Score_2013']).round(1)
print('Largest declines:', change.nsmallest(5, 'change')[['Economy', 'change']].to_dict('records'))
print('Largest improvements:', change.nlargest(5, 'change')[['Economy', 'change']].to_dict('records'))

fig = create_choropleth(
    change,
    location_col='REF_AREA',
    color_col='change',
    title='Change in overall freedom score, 2013 to 2026',
    colorbar_label='Change in score',
    range_color=(-40, 40),
)
fig.show()

Largest declines: [{'Economy': 'Tanzania', 'change': -38.0}, {'Economy': 'Nicaragua', 'change': -37.0}, {'Economy': 'El Salvador', 'change': -35.0}, {'Economy': 'Libya', 'change': -33.0}, {'Economy': 'Burkina Faso', 'change': -33.0}]
Largest improvements: [{'Economy': 'Fiji', 'change': 35.0}, {'Economy': 'Gambia, The', 'change': 28.0}, {'Economy': 'Bhutan', 'change': 23.0}, {'Economy': 'Sri Lanka', 'change': 20.0}, {'Economy': 'Kosovo', 'change': 19.0}]


### Interpretation

The change map is the global trend made geographic: a **red smear across most of Africa and the Americas** — deepest in Tanzania (−38), Nicaragua (−37), El Salvador (−35), Libya and Burkina Faso (−33) — versus **blue islands of improvement**: Fiji (+35), Gambia (+28), Bhutan (+23), Sri Lanka (+20) and Kosovo (+19). Europe is a patchwork of small declines; Oceania stands out as the only region that improved overall (notebook 06).

## 5. Map 4 — an indicator map: rule of law

**Question:** where is the world's weakest category weakest?

**Method:** choropleth of the rule-of-law subtotal (`FH_FIW_F`, 0–16) as % of scale for 2026 — the same normalisation used throughout notebooks 05–07.

In [6]:
sm_map = {'0_TO_4': 4, '0_TO_12': 12, '0_TO_16': 16, '0_TO_40': 40, '0_TO_60': 60, '0_TO_100': 100}

rol = long[(long['INDICATOR'] == 'FH_FIW_F') & (long['Year'] == 2026)].copy()
rol['Score'] = pd.to_numeric(rol['Score'], errors='coerce')
rol['pct'] = (rol['Score'] / rol['UNIT_MEASURE'].map(sm_map) * 100).round(1)
rol = rol.dropna(subset=['pct'])

fig = create_choropleth(
    rol,
    location_col='REF_AREA',
    color_col='pct',
    title='Rule of law (FH_FIW_F) as % of scale, 2026',
    colorbar_label='% of scale',
    range_color=(0, 100),
)
fig.show()

### Interpretation

The rule-of-law map shows the same dark belt as the overall map, but darker: **almost all of Africa and Asia sits below 40% of scale** on this dimension, with Europe and North America above 70%. Comparing this map with the overall-score map shows what notebook 07 found in numbers — rule of law is the weakest component nearly everywhere, and its geography is even more unforgiving than the headline score.

## 6. Map 5 — Africa zoom

**Question:** what does the continent look like on its own?

**Method:** the 2026 overall score with the map scope set to Africa — where the sub-region pattern from notebook 06 lives.

In [7]:
fig = create_choropleth(
    d2026,
    location_col='REF_AREA',
    color_col='Score',
    title='Overall freedom score in Africa, 2026',
    colorbar_label='Overall score (0-100)',
    range_color=(0, 100),
    scope='africa',
)
fig.show()

### Interpretation

The Africa zoom confirms notebook 06's sub-regional ranking geographically: **Southern Africa is the bright block** (South Africa, Botswana, Namibia above 70), **Western Africa is a mixed band**, and the dark core runs through **Northern, Middle and Eastern Africa** — Libya, Sudan, Chad, CAR and the EAC's lower scorers. Cabo Verde, Mauritius and the Seychelles shine as islands, exactly the top-10 leaders from notebook 05.

## 7. Map 6 — the status classification

**Question:** what is the geography of Free / Partly Free / Not Free?

**Method:** categorical choropleth of `FH_FIW_STATUS` for 2026 — the strings are used directly, never coerced to numbers.

In [8]:
status = long[(long['INDICATOR'] == 'FH_FIW_STATUS') & (long['Year'] == 2026)].dropna(subset=['Score'])
print('Status counts 2026:', status['Score'].value_counts().to_dict())

fig = create_choropleth(
    status,
    location_col='REF_AREA',
    color_col='Score',
    title='Status classification by economy, 2026',
    colorbar_label='Status',
    color_discrete_map={'F': '#2e8b57', 'PF': '#f0b400', 'NF': '#c0392b'},
)
fig.show()

Status counts 2026: {'F': 88, 'NF': 59, 'PF': 49}


### Interpretation

The status map mirrors the numeric maps: **green Free economies across Europe, the Americas, Oceania and the African leaders; red Not Free across the EAC, Central Asia, Russia, the Middle East and much of North Africa; amber Partly Free in the remaining band**. The 49 Partly Free economies include some of the world's largest — **India, Mexico and Singapore** — and the statuses split the world along exactly the lines the score maps drew.

## 8. East Africa zoom — Uganda, Kenya, Tanzania, Rwanda, Burundi

**Question:** what do the five East African economies look like on their own?

**Method:** the user-selected group `EAST_AFRICA_5` from `src/regions.py` (a subset of both the EAC and the UN M49 Eastern Africa sub-region), mapped with `fitbounds` so the view zooms to exactly these five economies. Three maps: the 2026 level, the change since 2013, and the freedom-of-expression question `D4` — the world's biggest decliner from notebook 07.

In [9]:
east = d2026[d2026['Economy'].isin(EAST_AFRICA_5)]
print(east[['Economy', 'Score']].sort_values('Score', ascending=False).to_string(index=False))

fig = create_choropleth(
    east,
    location_col='REF_AREA',
    color_col='Score',
    title='East Africa (Uganda, Kenya, Tanzania, Rwanda, Burundi): overall score 2026',
    colorbar_label='Overall score (0-100)',
    range_color=(0, 100),
    fitbounds=True,
)
fig.show()

 Economy  Score
   Kenya   49.0
  Uganda   33.0
Tanzania   28.0
  Rwanda   21.0
 Burundi   13.0


### Interpretation

The five neighbours span 36 points: **Kenya leads at 49**, then Uganda (33), Tanzania (28), Rwanda (21) and Burundi (13). The map makes the EAC ranking from notebook 05 concrete in space — even within these five economies, the region's internal gap is as wide as the gap between some whole continents.

In [10]:
east_change = change[change['Economy'].isin(EAST_AFRICA_5)]
print(east_change[['Economy', 'change']].sort_values('change').to_string(index=False))

fig = create_choropleth(
    east_change,
    location_col='REF_AREA',
    color_col='change',
    title='East Africa: change in overall score, 2013 to 2026',
    colorbar_label='Change in score',
    range_color=(-40, 40),
    fitbounds=True,
)
fig.show()

 Economy  change
Tanzania   -38.0
 Burundi   -21.0
  Uganda    -7.0
   Kenya    -6.0
  Rwanda    -3.0


### Interpretation

The change map shows how uneven the last 13 years were: **Tanzania (−38) and Burundi (−21) are the deep red**, while Rwanda (−3), Kenya (−6) and Uganda (−7) fell far less. Tanzania's collapse — from 66 in 2013 to 28 in 2026 — is the single biggest story on this map: the region's most populous economy ends barely above Burundi, its lowest neighbour.

In [11]:
d4_east = long[(long['INDICATOR'] == 'FH_FIW_D4') & (long['Year'] == 2026) & long['Economy'].isin(EAST_AFRICA_5)].copy()
d4_east['Score'] = pd.to_numeric(d4_east['Score'], errors='coerce')
d4_east = d4_east.dropna(subset=['Score'])
print(d4_east[['Economy', 'Score']].sort_values('Score', ascending=False).to_string(index=False))

fig = create_choropleth(
    d4_east,
    location_col='REF_AREA',
    color_col='Score',
    title='East Africa: freedom of expression (FH_FIW_D4), 2026',
    colorbar_label='Score (0-4)',
    range_color=(0, 4),
    fitbounds=True,
)
fig.show()

 Economy  Score
   Kenya    2.0
  Uganda    2.0
Tanzania    1.0
 Burundi    0.0
  Rwanda    0.0


### Interpretation

On the freedom-of-expression question, the five split again: **Kenya and Uganda score 2 of 4, Tanzania 1, and Rwanda and Burundi 0** — the two lowest overall scorers also sit at zero on the world's most-declining indicator (notebook 07). Even within five neighbours, expression rights are thin: no East African economy in this group reaches even half of the 0–4 scale.

## 9. From notebook to dashboard

**Question:** how will these maps become interactive?

**Method:** the notebook fixes key year/indicator combinations so the analysis is reproducible and comparable. The interactive selector — choose any **year** and any **indicator** and update the map live — is the dashboard's Map Explorer, built on the same `create_choropleth()` helper and the same processed dataset.

All nine maps share the same helper, the same colour conventions and the same source note, so the notebook and the dashboard tell exactly the same story.

## Summary — the pipeline complete

### What the maps added

- The two-tier world (notebook 06) is visible geographically: bright Europe/Oceania/Americas, dark Africa/Asia.
- The global decline is a red smear across Africa and the Americas with blue islands of improvement (Fiji, Gambia, Bhutan, Sri Lanka, Kosovo).
- Rule of law is weaker than the headline score on the map too — Africa and Asia sit below 40% of scale.
- Africa's bright Southern block vs its dark Northern/Middle/Eastern core matches the sub-regional analysis exactly.
- The categorical status splits the world along the same lines as the numeric scores — two consistent views.
- The **East Africa zoom** (Uganda, Kenya, Tanzania, Rwanda, Burundi) shows the region's internal span (Kenya 49 down to Burundi 13), Tanzania's −38 collapse, and expression scores of at most 2 of 4 — thin even among neighbours.

### The pipeline so far

01 profiling → 02 metadata/data dictionary → 03 cleaning (the long dataset) → 04 global trends → 05 country analysis → 06 regional analysis → 07 indicator analysis → 08 geographic analysis.

### Next question

*How do all of these analyses become an interactive dashboard?* — the Streamlit + Plotly dashboard (`dashboard/app.py`), which consumes only the processed long dataset and the reusable `src/` helpers.